# Liabilities analysis

Sibling project to `../expenses/`: same two-layer architecture, applied to `CRCData`
(Banco de Portugal credit register) from the proponent snapshot.

- **Layer 1** `liability_features(profile)`: full versioned feature vector (exposure,
  product mix, timeline, defaults, guarantor exposure). Consumers: the credit score
  model and layer 2.
- **Layer 2** `display_profile` / `render_profile_text` (pt-PT): the human-readable view.

Module: `liabilities_features.py` (stdlib only). Output: `liabilities_layers_output.csv`.

Design decisions:
1. Totals cover only entries where the proponent is **debtor** ("Devedor"); guarantor
   entries ("Avalista / fiador") are separate contingent exposure.
2. Reference date = `CRCData.DocumentDate`, so ages and "opened last 12 months" are
   point-in-time by construction.
3. Null `Installment` counts as 0 and is tallied (`n_missing_installment`): common on
   cards/overdrafts, so debt service is understated for card-heavy profiles.
4. `InstallmentType` is NOT used: it contains extraction garbage ("Legenda",
   "Data de Emissão:", ...).

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from liabilities_features import (FAMILIES, PRODUCT_FAMILY, display_profile,
                                  liability_features, render_profile_text)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
print(f"{len(snap)} proponents")

## 1. The liabilities table (one row per credit)

In [ ]:
liab_rows = []
for uuid, v in zip(snap["uuid"], snap["value"]):
    for liab in (json.loads(v).get("CRCData") or {}).get("Liabilities") or []:
        liab["uuid"] = uuid
        liab_rows.append(liab)
liab = pd.DataFrame(liab_rows)
liab["family"] = liab["Product"].map(lambda p: PRODUCT_FAMILY.get(p, "other"))

print(f"{len(liab)} liabilities, median {liab.groupby('uuid').size().median():.0f} per proponent")
print("\nby family (debtor + guarantor rows):")
print(liab.groupby("family")
      .agg(n=("family", "size"), total_debt=("TotalDebt", "sum"),
           total_default=("TotalDefault", "sum"))
      .round(0).sort_values("total_debt", ascending=False).to_string())

## 2. Layer 1: build the feature vector for every proponent

In [ ]:
feats = pd.DataFrame([
    {"uuid": u, **liability_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"feats: {feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

KEY = ["n_liabilities", "n_active", "total_debt", "total_installment",
       "revolving_debt_share", "oldest_credit_years", "n_opened_12m",
       "n_in_default", "total_default", "guarantor_n"]
print("\npopulation overview:")
print(feats[KEY].describe(percentiles=[.25, .5, .75, .9]).round(2).T.to_string())

In [ ]:
import matplotlib.pyplot as plt

INK, INK2, BLUE = "#0b0b0b", "#52514e", "#2a78d6"

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
d = feats["total_debt"].dropna()
axes[0].hist(d[d > 0], bins=40, color=BLUE, edgecolor="white", linewidth=0.8)
axes[0].set_xscale("log")
axes[0].set_title("Total debt per proponent (EUR, log)", loc="left", color=INK, fontsize=10)

fam_debt = pd.Series({f: feats[f"debt_{f}"].sum() for f in FAMILIES}).sort_values()
axes[1].barh(fam_debt.index, fam_debt.values / 1e6, color=BLUE)
axes[1].set_title("Debt by product family (EUR millions)", loc="left", color=INK, fontsize=10)

for ax in axes:
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(axis="x" if ax is axes[1] else "y", color="#e6e5e0", lw=0.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c3c2b7")
plt.tight_layout()
plt.show()

## 3. Layer 2: the human-readable profile

Three representative cases: clean, in default, and guarantor.

In [ ]:
fset = feats.set_index("uuid")
examples = [
    fset[(fset["n_in_default"] == 0) & (fset["guarantor_n"] == 0)].index[0],
    fset[(fset["n_in_default"] >= 2) & (fset["total_default"] > 2000)].index[0],
    fset[(fset["guarantor_n"] > 0) & (fset["guarantor_exposure"] > 1000)].index[0],
]
vmap = dict(zip(snap["uuid"], snap["value"]))
for u in examples:
    print(f"=== {u[:8]} ===")
    print(render_profile_text(liability_features(json.loads(vmap[u]))))
    print()

## 4. Validation: module vs independent pandas computation

Every core total recomputed from the raw liabilities table and compared. Must print 0.

In [ ]:
mism = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    vec = liability_features(json.loads(v))
    L = liab[liab["uuid"] == uuid]
    D = L[L["ResponsabilityType"] == "Devedor"]
    checks = {
        "total_debt": D["TotalDebt"].fillna(0).sum(),
        "total_installment": D["Installment"].fillna(0).sum(),
        "total_potential": D["Potential"].fillna(0).sum(),
        "n_active": (D["TotalDebt"].fillna(0) > 0).sum(),
        "n_in_default": (D["TotalDefault"].fillna(0) > 0).sum(),
        "total_default": D["TotalDefault"].fillna(0).sum(),
        "guarantor_n": (L["ResponsabilityType"] != "Devedor").sum(),
    }
    for k, expect in checks.items():
        if vec[k] is None or abs(float(vec[k]) - float(expect)) > 0.01:
            mism += 1
            print("MISMATCH", uuid[:8], k, vec[k], expect)
print(f"mismatches: {mism}")